# 📚 Belajar OOP, RAG, dan FastAPI Step-by-Step

Notebook ini mengajarkan **Object-Oriented Programming (OOP)** dari dasar,
kemudian secara bertahap membangun sistem **Retrieval-Augmented Generation (RAG)**
menggunakan **FastAPI** dan **FAISS**.

---

## Daftar Isi

| Bagian | Topik |
|--------|-------|
| 1 | Variabel dan Fungsi (review) |
| 2 | Apa itu Class dan Object? |
| 3 | Membuat Class Pertama: Mahasiswa |
| 4 | Attribute dan Method |
| 5 | Class Document untuk RAG |
| 6 | Encapsulation: Menyembunyikan Detail |
| 7 | Composition: Class Menggunakan Class Lain |
| 8 | Text Chunking |
| 9 | Embedding Sederhana |
| 10 | Vector Store dengan FAISS |
| 11 | Retriever dan Generator |
| 12 | Pipeline: Menyatukan Semuanya |
| 13 | Konsep Frontend-Backend |
| 14 | Latihan dan Eksperimen |

---

## Bagian 1: Variabel dan Fungsi (Review Singkat)

Sebelum masuk ke OOP, pastikan kamu sudah paham **variabel** dan **fungsi**.

In [ ]:
# === VARIABEL ===
# Variabel menyimpan data dengan tipe tertentu

nama = "Budi"           # string (teks)
umur = 21               # integer (bilangan bulat)
ipk = 3.75              # float (bilangan desimal)
aktif = True            # boolean (benar/salah)

print(f"Nama: {nama}, Umur: {umur}, IPK: {ipk}, Aktif: {aktif}")
print(f"Tipe 'nama': {type(nama).__name__}")
print(f"Tipe 'umur': {type(umur).__name__}")

In [ ]:
# === FUNGSI ===
# Fungsi: blok kode yang bisa dipanggil berulang kali

def hitung_nilai_akhir(tugas: float, uts: float, uas: float) -> float:
    """Menghitung nilai akhir mahasiswa."""
    return tugas * 0.3 + uts * 0.3 + uas * 0.4


# Panggil fungsi
nilai = hitung_nilai_akhir(tugas=85, uts=78, uas=90)
print(f"Nilai akhir: {nilai}")

### Masalah dengan Variabel Terpisah

Bagaimana kalau kita punya data 3 mahasiswa?

In [ ]:
# Tanpa OOP: variabel terpisah untuk setiap mahasiswa
nama1 = "Budi"
nim1 = "A001"
ipk1 = 3.75

nama2 = "Siti"
nim2 = "A002"
ipk2 = 3.90

nama3 = "Dedi"
nim3 = "A003"
ipk3 = 3.50

# Masalahnya:
# - Banyak variabel yang harus dikelola
# - Tidak ada hubungan jelas antara nama1 dan nim1
# - Bagaimana kalau ada 100 mahasiswa?

print("Bayangkan kalau ada 100 mahasiswa... 300 variabel? 😱")

---

## Bagian 2: Apa itu Class dan Object?

**Class** = cetakan/blueprint untuk membuat objek.  
**Object** = benda nyata yang dibuat dari cetakan tersebut.

Analogi:
- Class `KTP` = desain formulir KTP (template kosong)
- Object = KTP milik Budi yang sudah terisi data

Dengan class, kita bisa **mengelompokkan data dan fungsi** yang saling berhubungan.

---

## Bagian 3: Membuat Class Pertama — Mahasiswa

Berikut class `Mahasiswa` yang paling sederhana:

In [ ]:
class Mahasiswa:
    """Class yang mewakili seorang mahasiswa."""

    def __init__(self, nama: str, nim: str, ipk: float) -> None:
        """Inisialisasi: dipanggil saat membuat object baru.

        Parameter:
            nama: nama lengkap mahasiswa.
            nim: nomor induk mahasiswa.
            ipk: indeks prestasi kumulatif.
        """
        self.nama = nama   # attribute: menyimpan data
        self.nim = nim
        self.ipk = ipk


# Membuat object dari class Mahasiswa
mhs1 = Mahasiswa(nama="Budi", nim="A001", ipk=3.75)
mhs2 = Mahasiswa(nama="Siti", nim="A002", ipk=3.90)

# Akses attribute
print(f"Mahasiswa 1: {mhs1.nama} ({mhs1.nim}), IPK: {mhs1.ipk}")
print(f"Mahasiswa 2: {mhs2.nama} ({mhs2.nim}), IPK: {mhs2.ipk}")
print(f"\nSekarang data terorganisir: 1 object = 1 mahasiswa lengkap!")

### Penjelasan Kode:

| Bagian | Arti |
|--------|------|
| `class Mahasiswa:` | Mendefinisikan class baru bernama Mahasiswa |
| `def __init__(self, ...)` | Method khusus yang jalan otomatis saat object dibuat |
| `self` | Merujuk ke object yang sedang dibuat |
| `self.nama = nama` | Menyimpan parameter sebagai attribute object |
| `Mahasiswa(...)` | Membuat object baru (memanggil `__init__`) |

---

## Bagian 4: Attribute dan Method

**Attribute** = data yang disimpan di object.  
**Method** = fungsi yang bisa dilakukan oleh object.

In [ ]:
class Mahasiswa:
    """Class Mahasiswa dengan method."""

    def __init__(self, nama: str, nim: str, ipk: float) -> None:
        self.nama = nama
        self.nim = nim
        self.ipk = ipk

    def perkenalan(self) -> str:
        """Method: mengembalikan kalimat perkenalan."""
        return f"Halo, saya {self.nama} dengan NIM {self.nim}"

    def predikat(self) -> str:
        """Method: menentukan predikat berdasarkan IPK."""
        if self.ipk >= 3.75:
            return "Cum Laude"
        elif self.ipk >= 3.50:
            return "Sangat Memuaskan"
        else:
            return "Memuaskan"

    def to_dict(self) -> dict:
        """Method: mengubah object menjadi dictionary."""
        return {"nama": self.nama, "nim": self.nim, "ipk": self.ipk}


# Buat dan gunakan
mhs = Mahasiswa("Budi", "A001", 3.80)

print(mhs.perkenalan())      # Panggil method
print(f"Predikat: {mhs.predikat()}")
print(f"Dictionary: {mhs.to_dict()}")

---

## Bagian 5: Class Document — Mulai Masuk ke RAG

Sekarang kita mulai membuat class yang relevan dengan project RAG.
Class `Document` menyimpan satu dokumen teks yang akan diproses.

In [ ]:
class Document:
    """Mewakili satu dokumen yang masuk ke sistem RAG."""

    def __init__(self, title: str, content: str) -> None:
        self.title = title.strip()
        self.content = content.strip()

    def word_count(self) -> int:
        """Menghitung jumlah kata dalam dokumen."""
        return len(self.content.split())

    def preview(self, max_words: int = 10) -> str:
        """Mengambil beberapa kata pertama sebagai preview."""
        words = self.content.split()
        return " ".join(words[:max_words]) + "..."

    def to_dict(self) -> dict:
        """Mengubah object menjadi dictionary."""
        return {"title": self.title, "content": self.content}

    def __repr__(self) -> str:
        return f"Document(title='{self.title}', words={self.word_count()})"


# Buat dokumen
doc = Document(
    title="Pengenalan Python",
    content="Python adalah bahasa pemrograman yang mudah dipelajari. "
            "Python mendukung paradigma OOP dan banyak digunakan untuk "
            "data science, web development, dan artificial intelligence."
)

print(doc)                     # Otomatis panggil __repr__
print(f"Preview: {doc.preview()}")
print(f"Jumlah kata: {doc.word_count()}")

### Poin Penting:
- `__repr__` adalah method khusus agar saat kita `print(doc)` hasilnya informatif.
- `strip()` di `__init__` membersihkan spasi di awal/akhir teks.
- Method `to_dict()` berguna nanti saat mengirim data ke API.

---

## Bagian 6: Encapsulation — Menyembunyikan Detail

**Encapsulation** artinya menyembunyikan detail implementasi.
Pengguna class cukup tahu APA yang bisa dilakukan (method),
tanpa perlu tahu BAGAIMANA cara kerjanya di dalam.

Contoh: class `TextChunker` menyembunyikan logika chunking.

In [ ]:
class TextChunker:
    """Memecah teks menjadi potongan kecil (chunk)."""

    def __init__(self, chunk_size: int = 30, overlap: int = 5) -> None:
        """Parameter:
            chunk_size: jumlah kata per chunk.
            overlap: kata yang diulang antar chunk.
        """
        self.chunk_size = chunk_size
        self.overlap = overlap

    def split(self, text: str) -> list[str]:
        """Memecah teks menjadi list chunk."""
        words = text.split()
        if not words:
            return []

        step = self.chunk_size - self.overlap
        chunks = []

        for start in range(0, len(words), step):
            chunk = words[start : start + self.chunk_size]
            if not chunk:
                break
            chunks.append(" ".join(chunk))
            if start + self.chunk_size >= len(words):
                break

        return chunks


# Gunakan
chunker = TextChunker(chunk_size=10, overlap=2)

teks = ("Python adalah bahasa pemrograman tingkat tinggi yang dirancang "
        "untuk menjadi mudah dibaca dan ditulis oleh manusia")

chunks = chunker.split(teks)

print(f"Teks asli: {len(teks.split())} kata")
print(f"Dipecah menjadi {len(chunks)} chunk:\n")

for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk {i}: \"{chunk}\"")

### Kenapa Perlu Overlap?

Overlap memastikan konteks tidak terputus di batas chunk.
Misalnya kalimat "Python mudah dipelajari" mungkin terpotong tanpa overlap.
Dengan overlap, kata-kata di tepi akan muncul di kedua chunk.

---

## Bagian 7: Composition — Class Menggunakan Class Lain

**Composition** adalah konsep dimana satu class menggunakan object dari class lain.
Ini seperti merakit Lego: potongan kecil digabung menjadi sesuatu yang besar.

Contoh: class `DocumentProcessor` menggunakan `TextChunker` dan `Document`.

In [ ]:
class DocumentProcessor:
    """Memproses dokumen: memecahnya menjadi chunks.

    Ini contoh COMPOSITION: class ini pakai TextChunker di dalamnya.
    """

    def __init__(self, chunker: TextChunker) -> None:
        # Menyimpan object TextChunker sebagai attribute
        self.chunker = chunker

    def process(self, doc: Document) -> dict:
        """Memproses satu dokumen menjadi chunks."""
        chunks = self.chunker.split(doc.content)
        return {
            "title": doc.title,
            "total_words": doc.word_count(),
            "chunks": chunks,
            "total_chunks": len(chunks),
        }


# Buat components
chunker = TextChunker(chunk_size=8, overlap=2)
processor = DocumentProcessor(chunker=chunker)  # Composition!

doc = Document(
    title="Belajar OOP",
    content="OOP adalah paradigma pemrograman yang mengorganisasi kode "
            "ke dalam objek-objek yang memiliki data dan perilaku sendiri"
)

result = processor.process(doc)

print(f"Dokumen: {result['title']}")
print(f"Total kata: {result['total_words']}")
print(f"Total chunks: {result['total_chunks']}")
for i, chunk in enumerate(result['chunks'], 1):
    print(f"  [{i}] {chunk}")

---

## Bagian 8: Text Chunking pada Dokumen Nyata

Sekarang kita gunakan class `TextChunker` dari project asli
di folder `src/processing/chunker.py`.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.processing.chunker import TextChunker

# Buat chunker dengan parameter default project
chunker = TextChunker(chunk_size=30, overlap=5)

teks_panjang = """
Object-Oriented Programming atau OOP adalah paradigma pemrograman yang
mengorganisasi kode ke dalam objek-objek. Setiap objek adalah instance
dari sebuah class. Class adalah cetakan atau blueprint yang mendefinisikan
attribute dan method yang dimiliki objek. Dengan OOP, kode menjadi lebih
terstruktur, mudah dipelihara, dan bisa digunakan kembali. Empat pilar
utama OOP adalah encapsulation, inheritance, polymorphism, dan abstraction.
"""

chunks = chunker.split(teks_panjang)

print(f"Teks asli: {len(teks_panjang.split())} kata")
print(f"chunk_size={chunker.chunk_size}, overlap={chunker.overlap}")
print(f"Hasil: {len(chunks)} chunk\n")

for i, chunk in enumerate(chunks, 1):
    words = chunk.split()
    print(f"Chunk {i} ({len(words)} kata):")
    print(f"  {chunk}\n")

---

## Bagian 9: Embedding — Mengubah Teks ke Angka

Komputer tidak bisa langsung membandingkan teks. Kita perlu mengubah
teks menjadi **array angka** (disebut vector/embedding). Teks yang
bermakna mirip akan menghasilkan vector yang berdekatan.

Kita pakai pendekatan sederhana berbasis karakter (tanpa ML model).

In [ ]:
from src.processing.embedder import SimpleEmbedder

embedder = SimpleEmbedder(dimension=16)  # Pakai 16 dimensi agar mudah dilihat

# Langkah 1: Tokenisasi
teks = "Python adalah bahasa pemrograman"
tokens = embedder.clean_and_tokenize(teks)
print(f"Teks asli: '{teks}'")
print(f"Tokens: {tokens}\n")

# Langkah 2: Lihat bagaimana setiap kata dipetakan ke posisi
for word in tokens:
    index = embedder._word_to_index(word)
    print(f"  '{word}' -> posisi {index}")

# Langkah 3: Lihat vector embedding
vector = embedder.embed(teks)
print(f"\nVector embedding ({len(vector)} dimensi):")
print(f"  {vector}")
print(f"  Panjang vector (norm): {sum(v**2 for v in vector)**0.5:.4f}")

In [ ]:
# Bandingkan embedding dari teks yang mirip vs berbeda
import numpy as np

embedder = SimpleEmbedder(dimension=256)

teks_a = "Python adalah bahasa pemrograman yang populer"
teks_b = "Python merupakan bahasa pemrograman terkenal"  # Mirip dengan A
teks_c = "Kucing suka tidur di sofa"  # Sangat berbeda

vec_a = embedder.embed(teks_a)
vec_b = embedder.embed(teks_b)
vec_c = embedder.embed(teks_c)

# Hitung kemiripan (cosine similarity = dot product untuk vector ternormalisasi)
sim_ab = float(np.dot(vec_a, vec_b))
sim_ac = float(np.dot(vec_a, vec_c))

print(f"Teks A: '{teks_a}'")
print(f"Teks B: '{teks_b}'")
print(f"Teks C: '{teks_c}'")
print(f"\nKemiripan A-B: {sim_ab:.4f} (tinggi = mirip)")
print(f"Kemiripan A-C: {sim_ac:.4f} (rendah = berbeda)")
print(f"\n✅ Embedding sederhana kita bisa membedakan teks mirip vs berbeda!")

---

## Bagian 10: Vector Store dengan FAISS

FAISS menyimpan semua vector dan memungkinkan pencarian cepat.
Kita pakai `IndexFlatIP` (Inner Product) yang cocok untuk vector ternormalisasi.

In [ ]:
from src.storage.vector_store import FaissVectorStore

# Buat vector store
embedder = SimpleEmbedder(dimension=256)
store = FaissVectorStore(embedder=embedder)

# Tambahkan beberapa chunk
chunks_python = [
    "Python adalah bahasa pemrograman tingkat tinggi yang mudah dipelajari",
    "Python mendukung paradigma OOP dan pemrograman fungsional",
    "Library populer Python termasuk NumPy, Pandas, dan FastAPI",
]

chunks_java = [
    "Java adalah bahasa pemrograman yang berjalan di JVM",
    "Java sangat populer untuk aplikasi enterprise dan Android",
]

store.add_chunks("Panduan Python", chunks_python)
store.add_chunks("Panduan Java", chunks_java)

print(f"Total chunks tersimpan: {len(store.chunks)}")
print(f"Total vectors di FAISS: {store.index.ntotal}")

In [ ]:
# Cari chunk yang relevan
results = store.search("bahasa yang mudah dipelajari", top_k=3)

print("Pertanyaan: 'bahasa yang mudah dipelajari'\n")
print("Hasil pencarian:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['title']} (Chunk #{r['chunk_id']})")
    print(f"           {r['content'][:60]}...\n")

---

## Bagian 11: Retriever dan Generator

- **Retriever**: membungkus pencarian di vector store.
- **Generator**: menyusun jawaban dari konteks yang ditemukan.

In [ ]:
from src.retrieval.retriever import Retriever
from src.retrieval.generator import SimpleGenerator

# Buat retriever dan generator
retriever = Retriever(vector_store=store)
generator = SimpleGenerator()

# Proses pertanyaan
question = "Apa itu Python?"

# Step 1: Retriever mencari konteks
contexts = retriever.retrieve(question, top_k=2)
print(f"Pertanyaan: '{question}'")
print(f"Konteks ditemukan: {len(contexts)} chunk\n")

for ctx in contexts:
    print(f"  [{ctx['score']:.4f}] {ctx['content'][:50]}...")

# Step 2: Generator menyusun jawaban
answer = generator.generate(question, contexts)
print(f"\nJawaban: {answer}")

---

## Bagian 12: Pipeline — Menyatukan Semuanya

Class `RAGPipeline` adalah **orchestrator**: ia mengatur alur kerja
dari semua komponen. Pengguna cukup panggil `add_document()` dan `ask()`,
tanpa perlu tahu detail chunking, embedding, dll.

In [ ]:
from src.pipeline.rag_pipeline import RAGPipeline

# Buat pipeline
pipeline = RAGPipeline(chunk_size=25, overlap=5, dimension=256)

# Tambah beberapa dokumen
pipeline.add_document(
    title="Python OOP",
    content="Object-Oriented Programming atau OOP adalah paradigma pemrograman "
            "yang mengorganisasi kode ke dalam objek-objek. Setiap objek adalah "
            "instance dari sebuah class. Class mendefinisikan attribute dan method. "
            "Empat pilar OOP adalah encapsulation, inheritance, polymorphism, abstraction."
)

pipeline.add_document(
    title="FastAPI",
    content="FastAPI adalah framework web Python modern untuk membangun API. "
            "FastAPI menggunakan type hints untuk validasi otomatis dan menghasilkan "
            "dokumentasi Swagger secara otomatis. Performa FastAPI setara NodeJS."
)

pipeline.add_document(
    title="RAG",
    content="Retrieval-Augmented Generation atau RAG menggabungkan pencarian informasi "
            "dengan pembuatan jawaban. RAG mencari konteks relevan dari dokumen, "
            "lalu menyusun jawaban berdasarkan konteks tersebut. RAG mengurangi halusinasi."
)

# Lihat dokumen yang tersimpan
print("Dokumen tersimpan:")
for doc in pipeline.list_documents():
    print(f"  - {doc['title']}: {doc['content_length']} karakter, {doc['chunk_count']} chunk")

In [ ]:
# Tanya!
questions = [
    "Apa itu OOP?",
    "Bagaimana cara kerja FastAPI?",
    "Apa manfaat RAG?",
    "Bagaimana cara memasak nasi?"  # Tidak ada di dokumen
]

for q in questions:
    result = pipeline.ask(q)
    print(f"\n❓ {q}")
    print(f"💡 {result['answer'][:100]}...")
    print(f"   ({len(result['retrieved_contexts'])} konteks ditemukan)")

---

## Bagian 13: Konsep Frontend-Backend

Dalam arsitektur modern, aplikasi dibagi menjadi:

```
┌─────────────┐     HTTP Request      ┌─────────────┐
│  FRONTEND   │ ──────────────────────>│   BACKEND   │
│ (Streamlit) │<────────────────────── │  (FastAPI)  │
└─────────────┘     HTTP Response      └─────────────┘
     Tampilan                            Logika & Data
```

**Backend** (`api/main.py`):
- Menerima request HTTP (POST /ask, POST /documents)
- Memproses data menggunakan RAGPipeline
- Mengembalikan response JSON

**Frontend** (`frontend/streamlit_app.py`):
- Menampilkan form input untuk pengguna
- Mengirim data ke backend via HTTP
- Menampilkan hasil dari backend

**Keuntungan pemisahan ini:**
1. Frontend dan backend bisa dikembangkan terpisah
2. Backend bisa dipakai oleh banyak frontend (web, mobile, dll)
3. Masing-masing bisa di-deploy ke server yang berbeda

In [ ]:
# Simulasi bagaimana frontend berkomunikasi dengan backend
import json

# Simulasi: frontend mengirim request
request_body = json.dumps({"question": "Apa itu OOP?"})
print(f"Frontend mengirim POST /ask:")
print(f"  Body: {request_body}")

# Simulasi: backend memproses
parsed = json.loads(request_body)
result = pipeline.ask(parsed["question"])

# Simulasi: backend mengembalikan response
response = json.dumps({
    "question": result["question"],
    "answer": result["answer"],
    "contexts_count": len(result["retrieved_contexts"])
}, ensure_ascii=False, indent=2)

print(f"\nBackend mengembalikan response:")
print(response)

---

## Bagian 14: Latihan dan Eksperimen

Coba kerjakan latihan berikut untuk memperdalam pemahaman:

In [ ]:
# LATIHAN 1: Buat class Buku dengan attribute judul, penulis, halaman
# Tambahkan method: is_tebal() yang return True jika halaman > 300

# Tulis kode kamu di sini:
class Buku:
    pass  # Ganti ini dengan implementasi kamu


# Test:
# buku = Buku("Python Mastery", "Guido", 450)
# print(buku.is_tebal())  # Harusnya True

In [ ]:
# LATIHAN 2: Eksperimen dengan parameter chunking
# Coba ubah chunk_size dan overlap, amati hasilnya

from src.processing.chunker import TextChunker

teks = ("Dalam machine learning terdapat supervised learning dan unsupervised learning. "
        "Supervised learning memerlukan data berlabel sedangkan unsupervised learning "
        "mencari pola tanpa label. Contoh supervised adalah klasifikasi dan regresi. "
        "Contoh unsupervised adalah clustering dan dimensionality reduction.")

# Eksperimen: coba chunk_size=15 overlap=3
chunker_a = TextChunker(chunk_size=15, overlap=3)
print(f"chunk_size=15, overlap=3: {len(chunker_a.split(teks))} chunks")

# Coba juga: chunk_size=10, overlap=0 (tanpa overlap)
# Apa perbedaannya?
chunker_b = TextChunker(chunk_size=10, overlap=0)
print(f"chunk_size=10, overlap=0: {len(chunker_b.split(teks))} chunks")

In [ ]:
# LATIHAN 3: Tambahkan dokumen baru ke pipeline dan ajukan pertanyaan

pipeline.add_document(
    title="Machine Learning",
    content="Machine learning adalah cabang artificial intelligence yang memungkinkan "
            "komputer belajar dari data tanpa diprogram secara eksplisit. Terdapat "
            "tiga jenis utama: supervised learning, unsupervised learning, dan "
            "reinforcement learning. Supervised learning menggunakan data berlabel."
)

# Tanya sesuatu tentang machine learning
result = pipeline.ask("Apa jenis-jenis machine learning?")
print(f"Jawaban: {result['answer']}")

---

## 🎉 Selesai!

Kamu sudah mempelajari:

1. ✅ Variabel dan fungsi (dasar Python)
2. ✅ Class dan Object (dasar OOP)
3. ✅ Attribute dan Method
4. ✅ Encapsulation (menyembunyikan detail)
5. ✅ Composition (class pakai class lain)
6. ✅ Text Chunking (memecah dokumen)
7. ✅ Embedding (teks → vector)
8. ✅ Vector Store (FAISS)
9. ✅ Retriever dan Generator
10. ✅ Pipeline (orchestrator)
11. ✅ Konsep Frontend-Backend

### Langkah Selanjutnya:
1. Jalankan API: `uvicorn api.main:app --reload`
2. Buka dokumentasi: http://127.0.0.1:8000/docs
3. Jalankan frontend: `streamlit run frontend/streamlit_app.py`
4. Baca kode di folder `src/` dan perhatikan bagaimana setiap class punya file sendiri!